# Longor VLAbench launcher (`select_fruit`)

Runs the full Longor path (planner + executor + VLA service) on the default VLAbench task.

Launcher contract: `RUN_SCRIPT=examples/run_vlabench_task.py`

## 1 — Clone repo and capture secrets

In [ ]:
import getpass

github_token = getpass.getpass('GitHub PAT (repo scope): ')
openai_api_key = getpass.getpass('OpenAI API key: ')

In [ ]:
import os

if os.path.exists('/content/Longor'):
    os.chdir('/content/Longor')
    print('pulling repo...')
    !git pull
else:
    os.environ['CLONE_URL'] = f"https://{github_token}@github.com/pmish11/Longor-Robotics.git"
    !git clone $CLONE_URL "/content/Longor"
    os.chdir('/content/Longor')

## 2 — Headless rendering + app deps (VLAbench)

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

!apt-get update
!apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r deps/requirements-vlabench.txt
%pip install -e .

## 3 — SmolVLA policy venv

In [ ]:
!test -d /content/Longor/.vla-venv || python3 -m venv /content/Longor/.vla-venv
!/content/Longor/.vla-venv/bin/pip install -U pip
!/content/Longor/.vla-venv/bin/pip install -r deps/requirements-vla-smolvla-linux.txt
!/content/Longor/.vla-venv/bin/pip install -e .

## 4 — Configure `.env` for Longor run

In [ ]:
from pathlib import Path

Path('.env').write_text(
    f'OPENAI_API_KEY={openai_api_key}\n'
    'LOG_LEVEL=DEBUG\n'
    'LLM_MODEL=gpt-4.1-mini\n'
    'SAVE_VIDEO=1\n',
    encoding='utf-8',
)
print('.env written')

## 5 — Start VLA server + health

In [ ]:
%%bash
set -euo pipefail

cd /content/Longor
VLA_LOG=/tmp/longor_vlabench_vla_server.log
VLA_PYTHON=/content/Longor/.vla-venv/bin/python

if curl -fsS http://localhost:8765/health; then
  echo
  echo "VLA server already healthy"
  exit 0
fi

pkill -f "services/run_vla_server.py" || true
for _ in $(seq 1 60); do
  pgrep -f "services/run_vla_server.py" >/dev/null || break
  sleep 1
done
if pgrep -f "services/run_vla_server.py" >/dev/null; then
  echo "leftover VLA server still running; not starting another" >&2
  exit 1
fi

nohup "$VLA_PYTHON" services/run_vla_server.py >"$VLA_LOG" 2>&1 &

for _ in $(seq 1 360); do
  if curl -fsS http://localhost:8765/health; then
    echo
    echo "VLA server healthy"
    exit 0
  fi
  sleep 5
done

echo "VLA server not healthy; see $VLA_LOG" >&2
exit 1

## 6 — Run Longor once

In [ ]:
!python examples/run_vlabench_task.py --task "select_fruit"

## 7 — Stop VLA server

In [ ]:
!killall -vw python